# Task 5: Auto Tagging Support Tickets Using LLM

In [1]:
!pip install transformers datasets sentence-transformers scikit-learn torch accelerate -q

import nltk
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')

import numpy as np
import pandas as pd
import random
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer, util
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, pipeline, set_seed
)
from torch.utils.data import Dataset as TorchDataset

# Fix deprecation warning
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


Using device: cuda


In [2]:
# Categories & Templates
categories = ["billing", "technical", "account", "product_inquiry", "complaint", "feature_request"]

templates = {
    "billing": [
        "I was charged twice for my subscription on {date}. Please refund.",
        "My invoice #{inv} is incorrect, can you fix it?",
        "I didn't receive a receipt for my payment of ${amount}.",
        "Cancel my recurring payment immediately.",
        "The price changed without notice. Explain this charge."
    ],
    "technical": [
        "The app crashes every time I click on {feature}.",
        "I get error code {code} when launching.",
        "Syncing fails with message 'timeout'. Please help.",
        "The website is extremely slow since last update.",
        "Export to PDF produces corrupted files."
    ],
    "account": [
        "I forgot my password and reset link does not arrive.",
        "Can I change my email address associated with account?",
        "My account was locked after multiple login attempts.",
        "How to enable two-factor authentication?",
        "I want to delete my account and all data."
    ],
    "product_inquiry": [
        "Does your software support integration with {tool}?",
        "What is the maximum file size for upload?",
        "Is there a mobile version of the platform?",
        "Can I export data in CSV format?",
        "Are there API rate limits for enterprise plans?"
    ],
    "complaint": [
        "Your customer support is unresponsive for days.",
        "The latest update made the UI very confusing.",
        "I have been overcharged and nobody helps.",
        "Terrible experience with your service this month.",
        "Your chatbot is useless, I need a human agent."
    ],
    "feature_request": [
        "Please add dark mode to the web interface.",
        "It would be great to have keyboard shortcuts.",
        "Consider implementing bulk delete for items.",
        "I wish you had a calendar view for deadlines.",
        "Add a mobile app with offline support."
    ]
}

def generate_clean_ticket(category):
    template = random.choice(templates[category])
    template = template.replace("{date}", f"{random.randint(1,28)}/{random.randint(1,12)}/2024")
    template = template.replace("{inv}", str(random.randint(1000,9999)))
    template = template.replace("{amount}", str(random.randint(10,200)))
    template = template.replace("{feature}", random.choice(["save", "upload", "export", "login"]))
    template = template.replace("{code}", str(random.choice(["500", "404", "403", "Timeout"])))
    template = template.replace("{tool}", random.choice(["Slack", "Salesforce", "Jira", "Zoom"]))
    return template

def add_noise(text):
    if random.random() < 0.6 and len(text) > 5:
        idx = random.randint(0, len(text)-2)
        text = text[:idx] + text[idx+1] + text[idx] + text[idx+2:]
    return text

# Generate dataset
n_samples = 1500
data = []
for _ in range(n_samples):
    cat = random.choice(categories)
    clean_text = generate_clean_ticket(cat)
    if random.random() < 0.8:
        clean_text = add_noise(clean_text)
    if random.random() < 0.05:
        other_cats = [c for c in categories if c != cat]
        second_cat = random.choice(other_cats)
        if random.random() < 0.5:
            cat = second_cat
    data.append((clean_text, cat))

df = pd.DataFrame(data, columns=["ticket", "tag"])
train_df, test_df = train_test_split(df, test_size=0.2, stratify=df["tag"], random_state=42)
print(f"Train: {len(train_df)}, Test: {len(test_df)}")

Train: 1200, Test: 300


In [3]:
def evaluate_predictions(y_true, y_pred_proba_list, categories):
    top1_preds = [pred[0][0] for pred in y_pred_proba_list]
    top3_acc = sum(true in top3 for true, (top3, _) in zip(y_true, y_pred_proba_list)) / len(y_true)
    acc = accuracy_score(y_true, top1_preds)
    print(f"Top-1 Accuracy: {acc:.4f}")
    print(f"Top-3 Accuracy: {top3_acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, top1_preds, labels=categories))
    return acc, top3_acc

In [4]:
# Zero-shot
classifier = pipeline("zero-shot-classification", model="typeform/distilbert-base-uncased-mnli", device=0 if device=="cuda" else -1)

def zs_predict(text):
    res = classifier(text, categories, multi_label=False)
    if isinstance(res, list):
        res = res[0]
    return res['labels'][:3], res['scores'][:3]

zs_results = []
for t in test_df["ticket"]:
    top3, _ = zs_predict(t)
    zs_results.append((top3, None))

print("Zero-shot")
zs_acc, zs_top3 = evaluate_predictions(test_df["tag"].tolist(), zs_results, categories)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/258 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Zero-shot
Top-1 Accuracy: 0.3100
Top-3 Accuracy: 0.6233

Classification Report:
                 precision    recall  f1-score   support

        billing       0.69      0.24      0.35        46
      technical       0.00      0.00      0.00        43
        account       0.27      0.74      0.39        46
product_inquiry       0.88      0.13      0.23        52
      complaint       0.30      0.77      0.44        53
feature_request       0.00      0.00      0.00        60

       accuracy                           0.31       300
      macro avg       0.36      0.31      0.24       300
   weighted avg       0.35      0.31      0.23       300



In [5]:
# Few-shot
few_shot_examples = {cat: train_df[train_df["tag"]==cat]["ticket"].tolist()[:5] for cat in categories}

embedder = SentenceTransformer('all-MiniLM-L6-v2', device=device)
prototypes = {}
for cat, exs in few_shot_examples.items():
    if exs:
        emb = torch.stack([embedder.encode(e, convert_to_tensor=True).flatten() for e in exs])
        prototypes[cat] = emb.mean(dim=0)
    else:
        prototypes[cat] = embedder.encode("", convert_to_tensor=True).flatten()

def fs_predict(text):
    emb = embedder.encode(text, convert_to_tensor=True).flatten()
    sims = {cat: util.cos_sim(emb, proto).item() for cat, proto in prototypes.items()}
    sorted_cats = sorted(sims.items(), key=lambda x: x[1], reverse=True)
    return [c for c,_ in sorted_cats[:3]], [s for _,s in sorted_cats[:3]]

fs_results = [fs_predict(t) for t in test_df["ticket"]]
print("Few-shot")
fs_acc, fs_top3 = evaluate_predictions(test_df["tag"].tolist(), fs_results, categories)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Few-shot
Top-1 Accuracy: 0.8133
Top-3 Accuracy: 0.9433

Classification Report:
                 precision    recall  f1-score   support

        billing       0.77      0.96      0.85        46
      technical       0.61      0.93      0.73        43
        account       0.82      0.98      0.89        46
product_inquiry       0.98      0.83      0.90        52
      complaint       0.89      0.62      0.73        53
feature_request       0.95      0.65      0.77        60

       accuracy                           0.81       300
      macro avg       0.84      0.83      0.81       300
   weighted avg       0.85      0.81      0.81       300



In [6]:
# Fine-tuning
label2id = {c:i for i,c in enumerate(categories)}
id2label = {i:c for c,i in label2id.items()}

class TicketDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts; self.labels = labels; self.tokenizer = tokenizer; self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(), 'attention_mask': enc['attention_mask'].flatten(), 'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(categories), id2label=id2label, label2id=label2id)

train_labels = [label2id[t] for t in train_df["tag"]]
test_labels = [label2id[t] for t in test_df["tag"]]

train_dataset = TicketDataset(train_df["ticket"].tolist(), train_labels, tokenizer)
test_dataset = TicketDataset(test_df["ticket"].tolist(), test_labels, tokenizer)

training_args = TrainingArguments(
    output_dir="./results", num_train_epochs=3, per_device_train_batch_size=16,
    per_device_eval_batch_size=32, eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model="accuracy", report_to="none"
)

trainer = Trainer(model=model, args=training_args, train_dataset=train_dataset, eval_dataset=test_dataset,
                  compute_metrics=lambda p: {"accuracy": accuracy_score(p.label_ids, np.argmax(p.predictions, axis=1))})

trainer.train()

preds = trainer.predict(test_dataset)
probs = torch.nn.functional.softmax(torch.tensor(preds.predictions), dim=-1).numpy()
ft_results = []
for row in probs:
    top3 = np.argsort(row)[-3:][::-1]
    ft_results.append(([id2label[i] for i in top3], [row[i] for i in top3]))

print("Fine-tuned")
ft_acc, ft_top3 = evaluate_predictions(test_df["tag"].tolist(), ft_results, categories)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Fine-tuned
Top-1 Accuracy: 0.9633
Top-3 Accuracy: 0.9900

Classification Report:
                 precision    recall  f1-score   support

        billing       0.94      0.96      0.95        46
      technical       1.00      0.93      0.96        43
        account       0.96      1.00      0.98        46
product_inquiry       0.98      0.88      0.93        52
      complaint       0.93      1.00      0.96        53
feature_request       0.98      1.00      0.99        60

       accuracy                           0.96       300
      macro avg       0.96      0.96      0.96       300
   weighted avg       0.96      0.96      0.96       300



In [7]:
import pandas as pd
comparison = pd.DataFrame({
    "Method": ["Zero-shot", "Few-shot (5-shot)", "Fine-tuned (3 epochs)"],
    "Top-1 Accuracy": [zs_acc, fs_acc, ft_acc],
    "Top-3 Accuracy": [zs_top3, fs_top3, ft_top3]
})
comparison

,Method,Top-1 Accuracy,Top-3 Accuracy
0,Zero-shot,0.310000,0.623333
1,Few-shot (5-shot),0.813333,0.943333
2,Fine-tuned (3 epochs),0.963333,0.990000


In [8]:
# Top-3 predictions for first 5 test tickets
for i in range(5):
    print(f"\nTicket: {test_df.iloc[i]['ticket']}")
    print(f"True: {test_df.iloc[i]['tag']}")
    print(f"Zero-shot top3: {zs_results[i][0]}")
    print(f"Few-shot top3: {fs_results[i][0]}")
    print(f"Fine-tuned top3: {ft_results[i][0]}")


Ticket: Is there a mobile version of the platform?
True: product_inquiry
Zero-shot top3: ['product_inquiry', 'complaint', 'technical']
Few-shot top3: ['product_inquiry', 'feature_request', 'technical']
Fine-tuned top3: ['product_inquiry', 'technical', 'feature_request']

Ticket: Syncing fails with messaeg 'timeout'. Please help.
True: technical
Zero-shot top3: ['complaint', 'technical', 'account']
Few-shot top3: ['technical', 'feature_request', 'account']
Fine-tuned top3: ['technical', 'feature_request', 'product_inquiry']

Ticket: My account was locked after multiple login attepmts.
True: account
Zero-shot top3: ['account', 'complaint', 'billing']
Few-shot top3: ['account', 'complaint', 'technical']
Fine-tuned top3: ['account', 'product_inquiry', 'technical']

Ticket: I was charged twice for mys ubscription on 16/1/2024. Please refund.
True: billing
Zero-shot top3: ['account', 'product_inquiry', 'complaint']
Few-shot top3: ['billing', 'complaint', 'account']
Fine-tuned top3: ['billin